# BS-Detector: birth-scar validation against Carmen's annotations

This notebook compares the birth-scar detector with Carmen's hand annotations
from both validation packets (80 cells, 4 videos) and shows every cell side by
side: **Carmen's mark (red +)** next to the **detector's prediction**.

**Headline so far:** the new default detector puts **57/80 cells (71%) within
5 px** of Carmen's mark (median error 3.5 px). Held out by video or by packet it
gets 54/80 (68%), which is the fair estimate for new cells. The previous
detector got 16/80 (20%).

**To run:** Runtime → Change runtime type → **GPU**, then Runtime → Run all.
The packet H5 files must be in Google Drive (see the next cell). The first run
segments the 78 frames with Cellpose (~10–20 min on a Colab GPU); masks are
saved to Drive, so later runs take about a minute.

## 1. Setup

Edit `DATA_DIR` if the packets live elsewhere in your Drive. It must contain
`carmen_package_1/` and `carmen_package_2/`, each holding the four
`experiment_*.h5` files from `validation/h5_for_drive/`.

In [ ]:
# ── Edit if needed ───────────────────────────────────────────────────────────
DATA_DIR = '/content/drive/MyDrive/pombe_validation'     # holds carmen_package_1/ and carmen_package_2/
BRANCH   = 'codex/bs-detector-v2-redesign'               # change to 'main' once merged
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, subprocess
REPO_DIR = '/content/pombe-bs-detector'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH,
                    'https://github.com/widenerm/pombe-bs-detector.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Install Cellpose without letting pip replace Colab's CUDA build of torch.
import torch
with open('/tmp/constraints.txt', 'w') as f:
    f.write(f'torch=={torch.__version__}\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', '/tmp/constraints.txt',
                'cellpose==4.0.9', 'h5py', 'scikit-image'], check=True)

from google.colab import drive
drive.mount('/content/drive')
for packet in ('carmen_package_1', 'carmen_package_2'):
    n = len([f for f in os.listdir(os.path.join(DATA_DIR, packet)) if f.endswith('.h5')])
    print(f'{packet}: {n} H5 files')
    assert n == 4, f'Expected 4 H5 files in {DATA_DIR}/{packet}'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (segmentation will be slow)')

## 2. Segment the frames and collect Carmen's cells

Each annotated cell is found in the Cellpose mask and checked against an
independent identity anchor (the historical pole midpoint, not Carmen's scar
point). Masks are cached in `DATA_DIR/masks_*`, so this is fast after the first run.

In [ ]:
precision = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False) else 'float32'
CACHE = os.path.join(DATA_DIR, f'masks_{precision}')
CELLS = os.path.join(DATA_DIR, f'cells_{precision}.pkl')
subprocess.run([sys.executable, '-W', 'ignore', '-m', 'validation.scar_cells', '--cache', CACHE,
                '--out', CELLS, '--h5-root', DATA_DIR, '--segment-missing', '--precision', precision], check=True)

## 3. Accuracy of each detector mode

* **profile** (new default): scores every cross-section near each pole with a
  few named terms: expected scar distance from the new pole (scaled to cell
  size), the sibling-contact cue, and the width and ridge of the membrane traced
  in the brightfield image (the scar ridge is the widest point of the new end).
* **bilateral**: windowed curvature of the image-traced membrane on both flanks;
  `SCAR_BILATERAL_COMBINE = 'min'` requires both flanks to bulge, `'sum'` allows
  lopsided scars.
* **legacy**: the previous detector (curvature of the Cellpose outline).

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from validation.scar_eval import load, evaluate

records = load(CELLS)
MODES = {
    'profile (new default)':  {'SCAR_DETECTOR_MODE': 'profile'},
    'bilateral, min':         {'SCAR_DETECTOR_MODE': 'bilateral', 'SCAR_BILATERAL_COMBINE': 'min'},
    'bilateral, sum':         {'SCAR_DETECTOR_MODE': 'bilateral', 'SCAR_BILATERAL_COMBINE': 'sum'},
    'legacy (previous)':      {'SCAR_DETECTOR_MODE': 'legacy'},
}
results = {name: evaluate(records, overrides) for name, overrides in MODES.items()}

def summary(rows):
    e = np.array([np.inf if r['error'] is None else r['error'] for r in rows])
    n = len(e)
    return {'within 3 px': f'{(e <= 3).sum()}/{n}', 'within 5 px': f'{(e <= 5).sum()}/{n} ({(e <= 5).mean():.0%})',
            'within 10 px': f'{(e <= 10).sum()}/{n}', 'median error (px)': round(float(np.median(e)), 1)}

table = pd.DataFrame({name: summary(rows) for name, rows in results.items()}).T
display(table)
by_packet = pd.DataFrame({name: {pk: summary([r for r in rows if r['package'] == pk])['within 5 px']
                                 for pk in ('cr_v1', 'cr_v2')} for name, rows in results.items()}).T
by_packet.columns = ['packet 1, within 5 px', 'packet 2, within 5 px']
display(by_packet)

fig, ax = plt.subplots(figsize=(7, 4))
for name, rows in results.items():
    e = np.sort([r['error'] for r in rows])
    ax.step(e, np.arange(1, len(e) + 1) / len(e), where='post', label=name)
ax.axvline(5, color='k', ls=':')
ax.set(xlim=(0, 30), xlabel='distance from Carmen\'s mark (px)', ylabel='fraction of cells', title='Cumulative accuracy')
ax.legend(); plt.show()

The numbers above are measured on the same 80 cells the profile weights were
fitted to. Re-fitting with whole videos or whole packets held out gives 54/80
(68%) for profile mode; run `python -m validation.scar_fit <cells.pkl>` to
reproduce that.

## 4. Every cell: Carmen vs the detector

Red **+** = Carmen's mark; cyan **×** = profile (new default); yellow **·** =
legacy. Titles give the profile error in px. A ⚑ means the detector flagged
the cell for manual review (low confidence, or the scar is nearer the pole it
called old). Cells are sorted from largest to smallest error.

In [ ]:
prof = {r['key']: r for r in results['profile (new default)']}
leg  = {r['key']: r for r in results['legacy (previous)']}
order = sorted(records, key=lambda c: -prof[c['key']]['error'])

def short(video):   # experiment_03272025_wt_ymd494__xy3 -> 0327 xy3
    return video.split('_')[1][:4] + ' ' + video.split('__')[-1]

def show_cell(ax, c, half=70):
    r, l = prof[c['key']], leg[c['key']]
    img = c['crop'].astype(float); o = c['crop_origin']
    gy, gx = c['gt'] - o
    # keep both Carmen's mark and the prediction in view
    cy, cx = (c['gt'] + r['pred']) / 2 - o
    half = max(half, r['error'] / 2 + 30)
    ax.imshow(img, cmap='gray', vmin=np.percentile(img, 1), vmax=np.percentile(img, 99.5))
    ax.plot(c['contour'][:, 1] - o[1], c['contour'][:, 0] - o[0], color='w', lw=0.4, alpha=0.4)
    ax.plot(gx, gy, 'r+', ms=12, mew=2)
    ax.plot(r['pred'][1] - o[1], r['pred'][0] - o[0], 'cx', ms=10, mew=2)
    if l.get('pred') is not None:
        ax.plot(l['pred'][1] - o[1], l['pred'][0] - o[0], 'y.', ms=8)
    ax.set_xlim(cx - half, cx + half); ax.set_ylim(cy + half, cy - half); ax.axis('off')
    flag = ' ⚑' if r['debug'].get('scar_needs_review') or r['debug'].get('scar_pole_conflict') else ''
    ax.set_title(f"{short(c['video'])} f{c['frame']} {c['cell']}\n{r['error']:.1f} px{flag}",
                 fontsize=8, color='darkred' if r['error'] > 5 else 'black')

cols = 8
for start in range(0, len(order), 40):
    chunk = order[start:start + 40]
    fig, axs = plt.subplots(int(np.ceil(len(chunk) / cols)), cols, figsize=(2.3 * cols, 2.6 * np.ceil(len(chunk) / cols)))
    for ax in axs.flat: ax.axis('off')
    for ax, c in zip(axs.flat, chunk): show_cell(ax, c)
    plt.tight_layout(); plt.show()

## 5. Cells to discuss with Carmen

The cells where the detector and Carmen disagree by more than 5 px, with their
notes. Useful questions: is the mark on the scar they intended (newest scar,
right end)? Would they place it differently on a second look? Several have
notes such as "faint BS" or "septum maturing".

In [ ]:
rows = []
for c in order:
    r = prof[c['key']]
    if r['error'] <= 5: continue
    d = r['debug']
    rows.append({'video': short(c['video']), 'frame': c['frame'],
                 'cell': c['cell'], 'error px': round(r['error'], 1),
                 'flagged for review': bool(d.get('scar_needs_review') or d.get('scar_pole_conflict')),
                 'neighbor hint': r['hint'], 'Carmen notes': c['notes']})
display(pd.DataFrame(rows))

## 6. Look at one cell in detail

Set `CELL` to any (video, frame, cell name) from the gallery or table above. The
left panel shows the cell; the right panel shows how each candidate position
along the cell scored, with Carmen's position marked.

In [ ]:
CELL = ('0327 xy1', 77, 'G')   # (video as shown in the gallery, frame, cell)

c = next(c for c in records if short(c['video']) == CELL[0] and c['frame'] == CELL[1] and c['cell'] == CELL[2])
r = prof[c['key']]
cands = r['debug']['profile_candidates']
o = c['crop_origin']
gt_rel = c['gt'] - o
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1, 1.4]})
show_cell(a1, c, half=110)
for end, colour in ((0, 'tab:blue'), (1, 'tab:orange')):
    cs = [k for k in cands if k['end'] == end]
    if not cs: continue
    a2.plot([k['distance_px'] for k in cs], [k['score'] for k in cs], color=colour, label=f'from pole {end}')
    mids = np.array([np.mean(k['points'], axis=0) for k in cs])
    nearest = int(np.argmin(np.linalg.norm(mids - gt_rel, axis=1)))
    if np.linalg.norm(mids[nearest] - gt_rel) < 15:
        a2.axvline(cs[nearest]['distance_px'], color=colour, ls=':', label=f"Carmen's position (pole {end})")
a2.set(xlabel='distance from pole (px)', ylabel='score', title='Score of each candidate cross-section')
a2.legend(); plt.show()
print('Score terms at the chosen position:', {k: round(v, 2) for k, v in r['debug']['scar_terms'].items()})
print('Confidence:', round(r['debug']['scar_confidence'], 2), '| flagged:', r['debug']['scar_needs_review'])

## Where things stand

* Main finding: the Cellpose outline smooths the scar away. The signal is the
  membrane line in the brightfield image, which the new detector traces
  directly.
* Remaining misses are consistent across neighbouring frames, so they are
  systematic, not noise. Carmen's own marks on the same cell in adjacent frames
  differ by 1–4 px, which matters at a 5 px threshold.
* The most useful next steps on the annotation side:
  1. a second person annotating the same 80 cells, to measure the noise ceiling;
  2. Carmen re-checking the cells in section 5;
  3. a third packet focused on older cells, where the scar sits 40–65 px from
     the pole, and cells with no neighbouring sibling.

Full report: `validation/reports/scar_v3/REPORT.md` in the repository.